# 01 - Extract LOCA2 6 km (CMIP6, Scripps)

Daily `pr`, `tasmax`, `tasmin`, `wspeed` for the configured GCM ensemble, scenarios
SSP2-4.5 and SSP3-7.0 plus historical 1950-2014, subset to the Tahoe bbox.

**Source (verified 2026-07-29): the Cal-Adapt `cadcat` S3 mirror** -
`s3://cadcat/loca2/ucsd/<model>/<scenario>/<member>/day/<var>/d03` Zarr stores on a
western-US domain (lat 29.6-45.0, lon -128.4..-111.0 - covers all of NV-side Tahoe).
Anonymous access; Zarr chunking means only the bbox window transfers. The cirrus
HTTP region files remain a documented fallback (`fallback_http` in config) - measured
~3.3 MB/s there vs seconds per store here. Member fallback: where the configured
member lacks a scenario (MPI-ESM1-2-HR ssp245), the code walks r1i1p1f1, r2i1p1f1.

In [1]:
import sys
print("Python:", sys.executable)

import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# Pipeline root = climate/ (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.io import load_config, get_logger, append_manifest, sha256_file

cfg = load_config()
log = get_logger("01_extract_loca2")

RAW = ROOT / cfg["paths"]["raw"]
PROCESSED = ROOT / cfg["paths"]["processed"]
OUTPUTS = ROOT / cfg["paths"]["outputs"]
for p in (RAW, PROCESSED, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

BBOX = cfg["study_area"]["bbox"]
log.info(f"bbox: lon {BBOX['lon_min']}..{BBOX['lon_max']}, lat {BBOX['lat_min']}..{BBOX['lat_max']}")

Python: C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe


2026-07-30 08:13:06 | INFO | 01_extract_loca2 | Log file: C:\Users\mbindl\Documents\GitHub\PROTECT\climate\logs\01_extract_loca2_2026-07-30_081306.log


2026-07-30 08:13:06 | INFO | 01_extract_loca2 | bbox: lon -120.5..-119.5, lat 38.5..39.5


## Discover stores
List actual members per model x scenario on S3 and build the pull plan.

In [2]:
import s3fs

L = cfg["sources"]["loca2"]
FS = s3fs.S3FileSystem(anon=True)
PREFIX = L["s3_prefix"]
DOMAIN = L["domain"]


def pick_member(model, scenario):
    """Prefer the configured member, then r1i1p1f1, r2i1p1f1, then first listed."""
    preferred = L.get("member_overrides", {}).get(model, L["member"])
    try:
        have = [k.rsplit("/", 1)[1] for k in FS.ls(f"{PREFIX}/{model.lower()}/{scenario}")]
    except FileNotFoundError:
        return None
    for m in [preferred, "r1i1p1f1", "r2i1p1f1"] + sorted(have):
        if m in have:
            if m != preferred:
                log.info(f"[{model}] {scenario}: using fallback member {m}")
            return m
    return None


plan = []
missing = []
for gcm in L["gcms"]:
    for scen in ["historical"] + L["scenarios"]:
        member = pick_member(gcm, scen)
        if member is None:
            log.warning(f"[{gcm}] {scen}: no members on S3 - skipped")
            continue
        for var in L["variables"]:
            store = f"{PREFIX}/{gcm.lower()}/{scen}/{member}/day/{var}/{DOMAIN}"
            # not every model publishes every variable (e.g. FGOALS-g3 has no wind)
            if not FS.exists(store):
                missing.append(f"{gcm}/{scen}/{var}")
                continue
            plan.append({"gcm": gcm, "scenario": scen, "variable": var,
                         "member": member, "store": store})
plan = pd.DataFrame(plan)
plan.to_csv(OUTPUTS / "loca2_plan.csv", index=False)
if missing:
    log.info(f"{len(missing)} model x scenario x variable cells not published on S3 "
             f"(documented gap, not an error): {missing}")
log.info(f"{len(plan)} stores planned across {plan['gcm'].nunique() if len(plan) else 0} GCMs")
plan.head(12)

2026-07-30 08:14:22 | INFO | 01_extract_loca2 | [MPI-ESM1-2-HR] ssp245: using fallback member r1i1p1f1


2026-07-30 08:14:23 | INFO | 01_extract_loca2 | 3 model x scenario x variable cells not published on S3 (documented gap, not an error): ['FGOALS-g3/historical/wspeed', 'FGOALS-g3/ssp245/wspeed', 'FGOALS-g3/ssp370/wspeed']


2026-07-30 08:14:23 | INFO | 01_extract_loca2 | 57 stores planned across 5 GCMs


,gcm,scenario,variable,member,store
0,ACCESS-CM2,historical,pr,r1i1p1f1,cadcat/loca2/ucsd/access-cm2/historical/r1i1p1...
1,ACCESS-CM2,historical,tasmax,r1i1p1f1,cadcat/loca2/ucsd/access-cm2/historical/r1i1p1...
2,ACCESS-CM2,historical,tasmin,r1i1p1f1,cadcat/loca2/ucsd/access-cm2/historical/r1i1p1...
3,ACCESS-CM2,historical,wspeed,r1i1p1f1,cadcat/loca2/ucsd/access-cm2/historical/r1i1p1...
4,ACCESS-CM2,ssp245,pr,r1i1p1f1,cadcat/loca2/ucsd/access-cm2/ssp245/r1i1p1f1/d...
5,ACCESS-CM2,ssp245,tasmax,r1i1p1f1,cadcat/loca2/ucsd/access-cm2/ssp245/r1i1p1f1/d...
6,ACCESS-CM2,ssp245,tasmin,r1i1p1f1,cadcat/loca2/ucsd/access-cm2/ssp245/r1i1p1f1/d...
7,ACCESS-CM2,ssp245,wspeed,r1i1p1f1,cadcat/loca2/ucsd/access-cm2/ssp245/r1i1p1f1/d...
8,ACCESS-CM2,ssp370,pr,r1i1p1f1,cadcat/loca2/ucsd/access-cm2/ssp370/r1i1p1f1/d...
9,ACCESS-CM2,ssp370,tasmax,r1i1p1f1,cadcat/loca2/ucsd/access-cm2/ssp370/r1i1p1f1/d...


## Subset and save
Open each Zarr store lazily, slice the bbox, load only that window, write
`data/raw/loca2/<var>.<GCM>.<scenario>.<member>.d03__tahoe.nc`. Skips files already
present unless `run.overwrite_downloads`.

In [3]:
def bbox_slice(ds):
    lon_min, lon_max = BBOX["lon_min"], BBOX["lon_max"]
    if float(ds.lon.max()) > 180:          # 0-360 convention
        lon_min, lon_max = lon_min + 360, lon_max + 360
    lat_asc = bool(ds.lat[0] < ds.lat[-1])
    lat_sl = slice(BBOX["lat_min"], BBOX["lat_max"]) if lat_asc else \
             slice(BBOX["lat_max"], BBOX["lat_min"])
    return ds.sel(lon=slice(lon_min, lon_max), lat=lat_sl)


import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

loca2_dir = RAW / "loca2"
loca2_dir.mkdir(exist_ok=True)
MANIFEST_LOCK = threading.Lock()   # append_manifest is read-modify-write


def open_store(store):
    """Fall back for stores lacking consolidated .zmetadata."""
    try:
        return xr.open_zarr(FS.get_mapper(store), consolidated=True)
    except KeyError:
        return xr.open_zarr(FS.get_mapper(store), consolidated=False)


def fetch_one(row):
    out = loca2_dir / (f"{row['variable']}.{row['gcm']}.{row['scenario']}"
                       f".{row['member']}.{DOMAIN}__tahoe.nc")
    if out.exists() and not cfg["run"]["overwrite_downloads"]:
        return "skipped", out.name
    try:
        ds = open_store(row["store"])
        sub = bbox_slice(ds).load()
        if min(sub.sizes.get("lat", 0), sub.sizes.get("lon", 0)) == 0:
            return "failed", f"empty bbox slice: {row['store']}"
        # xarray decodes to float64; LOCA2 is native float32 - cast to halve size
        sub = sub.astype({v: "float32" for v in sub.data_vars})
        sub.to_netcdf(out, encoding={v: {"zlib": True, "complevel": 4}
                                     for v in sub.data_vars})
        with MANIFEST_LOCK:
            append_manifest({"file": str(out.relative_to(ROOT)),
                             "source_url": f"s3://{row['store']}",
                             "size_bytes": out.stat().st_size, "sha256": sha256_file(out),
                             "retrieved_date": str(pd.Timestamp.today().date()),
                             "notebook": "01_extract_loca2"})
        return "done", f"{out.name} ({out.stat().st_size/1e6:.1f} MB)"
    except Exception as e:
        return "failed", f"{row['store']}: {e}"


counts = {"done": 0, "skipped": 0, "failed": 0}
if len(plan):
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = [pool.submit(fetch_one, row) for _, row in plan.iterrows()]
        for fut in as_completed(futures):
            status, msg = fut.result()
            counts[status] += 1
            if status == "done":
                log.info(f"[{counts['done']}] {msg}")
            elif status == "failed":
                log.warning(f"FAILED {msg}")

log.info(f"LOCA2 subsets: {counts['done']} downloaded, "
         f"{counts['skipped']} already present, {counts['failed']} failed")

2026-07-30 08:14:23 | INFO | 01_extract_loca2 | LOCA2 subsets: 0 downloaded, 57 already present, 0 failed
